# Chapter 28 — MongoDB-to-pandas normalization and comparing storage backends

**Book role:** Databases  
**Suggested book space:** about 9 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Flatten nested documents into tabular schemas.
- Handle ObjectId, arrays, Decimal128, and dates deliberately.
- Compare CSV, Excel, MySQL, and MongoDB adapter behavior.
- Define a source-independent analytics contract.

## 1. Nested documents are not automatically rectangular data

A DataFrame expects values organized into labeled columns. MongoDB documents may contain nested objects, arrays, optional attributes, and schema variants. `pd.json_normalize()` can flatten nested dictionaries with a separator, but an array still requires a policy: keep it as a list, expand it into multiple rows, aggregate it, or move it to a related table. Every choice changes what one row means.

Flattening should be a defined schema transformation, not a generic “flatten everything” button. Preserve identifiers, specify column names and data types, detect collisions such as both `customer.id` and a literal key named `customer_id`, and test sparse fields. Record when a nested array expansion multiplies rows so downstream metrics do not accidentally double-count orders.

### Worked example 1: Normalize nested documents

This example demonstrates two grains: one row per order and one row per line item. Aggregating orders after expanding items requires care because order-level values may repeat.

In [ ]:
import pandas as pd

records = [
    {"order_id": "A1", "region": "North", "customer": {"id": "C10", "tier": "Gold"}, "items": [{"sku": "P1", "qty": 2}]},
    {"order_id": "A2", "region": "West", "customer": {"id": "C11", "tier": "Silver"}, "items": [{"sku": "P2", "qty": 1}, {"sku": "P3", "qty": 4}]},
]
orders = pd.json_normalize(records, sep="_")
print("One row per order:")
print(orders[["order_id", "region", "customer_id", "customer_tier"]])

items = pd.json_normalize(records, record_path="items", meta=["order_id", "region"])
print("One row per order item:")
print(items)

## 2. Type normalization and precision

MongoDB’s ObjectId is not a numeric identifier; it can be displayed as its hexadecimal string while preserving the original value in the adapter. BSON date values should become timezone-aware or deliberately naive pandas timestamps according to the application’s time policy. Decimal128 is intended for decimal data and should not be blindly converted to binary float when exact monetary precision is required.

A consistent analysis contract can use a string order ID, a normalized datetime, category/region strings, numeric quantities, and a defined money representation. That contract may choose decimal arithmetic or carefully tested numeric conversion. Document rounding rules. Display formatting is separate from internal numeric representation and should happen in the view/export layer.

### Worked example 2: Normalize monetary values explicitly

Financial arithmetic often benefits from Decimal until a documented conversion point. Plotting libraries typically use floating-point arrays; do not silently treat display-oriented conversion as an accounting ledger calculation.

In [ ]:
from decimal import Decimal
import pandas as pd

# Illustrative values from a BSON Decimal128 conversion or a JSON payload.
raw_values = [Decimal("10.25"), Decimal("0.10"), Decimal("1000.00")]
exact_total = sum(raw_values, Decimal("0.00"))
print("Exact decimal total:", exact_total)

# A float representation is sometimes useful for plotting, but make that conversion explicit.
plot_values = pd.Series([float(value) for value in raw_values], dtype="float64")
print("Plotting series:", plot_values.tolist())

## 3. Cross-source equivalence and source metadata

CSV, Excel, SQL, and MongoDB do not share identical guarantees. SQL can enforce constraints and transactions; Excel allows arbitrary formulas, cell types, and layouts; CSV is untyped; MongoDB allows documents with varied fields. A common adapter should normalize the values needed for analysis and report what cannot be normalized, rather than pretending that all sources behave identically.

For reproducible analytics, keep a source descriptor: type, location label (not secret URI), extraction time, selected range/query name, source row count, normalized row count, rejected count, and schema version. This allows an export to explain where data came from without exposing credentials or private connection details.

## Guided lab: Lab: source equivalence test

Load equivalent tiny records from CSV, an Excel sheet, a MySQL query, and a MongoDB collection. Normalize each source into a documented schema and compare key columns, row grain, totals, null behavior, and date handling. Write a test that proves the same metric is produced across sources, with explicit tolerances only where conversion is justified.

## Review questions and practice

1. What can happen to row count when nested arrays are expanded?
2. Why should ObjectId not be treated as a measure?
3. Why can Decimal-to-float conversion matter for money?
4. What metadata makes an analysis reproducible without exposing secrets?

## Further reading

- pandas json_normalize: https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html
- Python Decimal: https://docs.python.org/3/library/decimal.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.